In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=HOyEpF5uxdvXGKEtNvLI5cAisIp48a&access_type=offline&code_challenge=RCeTt9bt3wfrwy598Jyi2ZSqV5ykuP5HFS8J-wpV99c&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame
from gentropy.dataset.study_locus import StudyLocus
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics

from gentropy.susie_finemapper import SusieFineMapperStep

Loading BokehJS ...

In [2]:

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/09 14:12:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/09 14:12:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
path_to_release_folder="gs://open-targets-data-releases/25.03/"
#path_to_release_folder="gs://open-targets-pre-data-releases/24.12-uo_test-3/output/genetics/parquet/"
#path_to_release_folder="gs://ot_orchestration/releases/25.02_freeze1/"

si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

In [4]:
from gentropy.dataset.colocalisation import Colocalisation
from gentropy.dataset.l2g_feature_matrix import L2GFeatureMatrix
from gentropy.dataset.l2g_gold_standard import L2GGoldStandard
from gentropy.dataset.l2g_prediction import L2GPrediction
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.target_index import TargetIndex
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.l2g.feature_factory import L2GFeatureInputLoader
from gentropy.method.l2g.model import LocusToGeneModel
from gentropy.method.l2g.trainer import LocusToGeneTrainer

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



In [5]:
credible_set_path=path_to_release_folder+"output/credible_set/"

#colocalisation_path=["gs://open-targets-data-releases/25.03/output/colocalisation_coloc",
#"gs://open-targets-data-releases/25.03/output/colocalisation_ecaviar"]

colocalisation_path="gs://open-targets-data-releases/25.03/output/colocalisation_coloc"

credible_set = StudyLocus.from_parquet(
    session, credible_set_path, recursiveFileLookup=True
)
coloc = (
    Colocalisation.from_parquet(
        session, colocalisation_path, recursiveFileLookup=True
    )
    if colocalisation_path
    else None
)

In [6]:
credible_set.df.show(1)

25/05/09 14:13:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+---------+---------------+----------+--------+------+------------+-------+------+--------------+--------------+-------------------------------+-------------+-------------------+--------------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+--------------------+--------------------+--------------------+----------+
|        studyLocusId|studyType|      variantId|chromosome|position|region|     studyId|   beta|zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|     qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|locusStart|locusEnd|sampleSize|               ldSet|               locus|          confidence|isTransQtl|
+--------------------+---------+---------------+----------+--------+------+------------+-------+------+--------------+--------------+-------------------------------+-------------+-----------

In [7]:
coloc.df.show(1)

+--------------------+--------------------+--------------+----------+--------------------+--------------------------+--------------------+--------------------+--------------------+-------------------+------------------+----+--------------------+
|    leftStudyLocusId|   rightStudyLocusId|rightStudyType|chromosome|colocalisationMethod|numberColocalisingVariants|                  h0|                  h1|                  h2|                 h3|                h4|clpp|betaRatioSignAverage|
+--------------------+--------------------+--------------+----------+--------------------+--------------------------+--------------------+--------------------+--------------------+-------------------+------------------+----+--------------------+
|00103a2c9375268ff...|8f6164936023652f4...|          eqtl|        11|               COLOC|                        12|7.642090398173559...|6.849524791401351E-7|1.475080232255407...|0.13134122289216718|0.8686580921553568|NULL|                -1.0|
+---------------

In [8]:
df=credible_set.df.select("studyLocusId","isTransQtl")

In [9]:
coloc.df.count()

24976024

In [13]:
df1=coloc.df.join(df,df.studyLocusId==coloc.df.rightStudyLocusId,how="inner")
#df1.count()

In [11]:
df1.show(1)

+--------------------+--------------------+--------------+----------+--------------------+--------------------------+--------------------+--------------------+--------------------+-------------------+-----------------+----+--------------------+--------------------+----------+
|    leftStudyLocusId|   rightStudyLocusId|rightStudyType|chromosome|colocalisationMethod|numberColocalisingVariants|                  h0|                  h1|                  h2|                 h3|               h4|clpp|betaRatioSignAverage|        studyLocusId|isTransQtl|
+--------------------+--------------------+--------------+----------+--------------------+--------------------------+--------------------+--------------------+--------------------+-------------------+-----------------+----+--------------------+--------------------+----------+
|958327a14efb17512...|0003272064af24c99...|         tuqtl|         3|               COLOC|                        74|9.122288422162504E-9|2.492225434896531E-5|3.04322553

In [12]:
df1=df1.filter((f.col("isTransQtl")==False) | (f.col("isTransQtl").isNull()))
df1.count()

23448127

In [15]:
df1 = df1.filter(
                (~f.col("isTransQtl")) | (f.col("isTransQtl").isNull())
            )
df1.count()

23448127

In [16]:
df1=df1.drop("studyLocusId", "isTransQtl")

In [ ]:
coloc = Colocalisation(
    _df=df1,
    _schema=Colocalisation.get_schema(),
)

25/05/09 18:24:58 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 405279 ms exceeds timeout 120000 ms
25/05/09 18:24:58 WARN SparkContext: Killing executors is not supported by current scheduler.
25/05/09 18:25:03 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$